# Module 1.3 — Libraries, Command-Line Tools, Web APIs, and Unit Testing

## Data Science Bootcamp | Interactive Code-Along

> **Today’s outcome:** You will reuse code through imports, understand how Python scripts receive command-line arguments, request and inspect data from a web API, and write small automated tests for your functions.

**Suggested duration:** 3.5–4.5 hours, including demonstrations, code-alongs, independent practice, and debrief.

### Prerequisites
Complete Modules 1.1 and 1.2, or be comfortable with functions, strings, lists, dictionaries, conditionals, loops, and `try` / `except`.

### Learning objectives
By the end of this module, you can:
- Explain the difference between a module, package, library, and virtual environment.
- Import standard-library code using `import` and `from ... import ...`.
- Generate repeatable random values with a seeded random-number generator.
- Explain `sys.argv` and safely inspect command-line arguments.
- Describe the roles of `pip`, `requirements.txt`, and a virtual environment.
- Explain the request-response model and common HTTP status codes.
- Retrieve, decode, inspect, and validate JSON-style API data.
- Write `assert` tests for normal, boundary, and invalid cases.
- Organize a basic `pytest` test file outside a notebook.

### Why this matters in data science
Most data-science work depends on libraries, external data sources, reproducible environments, and trustworthy functions. A model pipeline is only as reliable as the code and data it receives.

## 1. Reusing code with modules and libraries

Python comes with a **standard library**: useful modules installed with Python itself. Other developers publish third-party packages that can be installed with `pip`.

| Term | Meaning | Example |
|---|---|---|
| Module | One Python file containing reusable code | `random`, `json`, `sys` |
| Package | A collection of related modules | `requests`, `pandas` |
| Library | General term for reusable code | Python standard library or pandas |
| Virtual environment | Isolated set of project dependencies | `.venv` |

### Two common import styles

```python
import random
random.choice(['A', 'B'])

from random import choice
choice(['A', 'B'])
```

For teaching and production readability, prefer `import module_name` when it makes the source of a function clear. Avoid wildcard imports such as `from module import *`.

In [ ]:
import random

study_topics = ['Python', 'SQL', 'Pandas', 'Statistics', 'Machine Learning']
selected_topic = random.choice(study_topics)
practice_minutes = random.randint(20, 60)

print(f'Today: study {selected_topic} for {practice_minutes} minutes.')

### Reproducible randomness
Randomness is useful for simulations, sampling, train/test splitting, and demonstrations. A seed makes a pseudorandom sequence repeatable, which helps you debug and reproduce results.

In real projects, record the seed used for experiments. Do not use predictable pseudorandom generators from `random` for security-sensitive tasks such as passwords or tokens.

In [ ]:
import random

random.seed(42)
sample_a = random.sample(range(1, 101), k=5)

random.seed(42)
sample_b = random.sample(range(1, 101), k=5)

print('First sample: ', sample_a)
print('Second sample:', sample_b)
print('Are they identical?', sample_a == sample_b)

### Checkpoint 1
Import `random` and create a small program that chooses one dataset theme from a list and generates a random integer from 100 through 500 to represent a sample size. Run it twice, then add a seed and run it twice again. What changes?

In [ ]:
# Your code here
# dataset_themes = ['retail', 'health', 'transport', 'education']

## 2. Scripts and command-line arguments

A notebook is useful for exploration. A script is useful when you want a repeatable program you can run from a terminal, scheduler, or automation pipeline.

When you run a script such as:

```bash
python report.py sales.csv 2026-09
```
Python stores command-line values in `sys.argv`. The first item, `sys.argv[0]`, is normally the script path. Values supplied after the script name begin at `sys.argv[1]`.

Because `sys.argv` is a list, accessing an argument that was not supplied can cause an `IndexError`. Validate the number of arguments before reading them.

In [ ]:
# This notebook-safe simulation represents: python report.py sales.csv 2026-09
simulated_argv = ['report.py', 'sales.csv', '2026-09']

script_name = simulated_argv[0]
arguments = simulated_argv[1:]

print('Script name:', script_name)
print('Arguments:', arguments)
print('Input file:', arguments[0])
print('Reporting period:', arguments[1])

In [ ]:
def parse_report_arguments(argv):
    """Validate simulated sys.argv values for a simple reporting script."""
    if len(argv) != 3:
        return 'Usage: python report.py <input_file> <period>'

    input_file = argv[1]
    period = argv[2]
    return {'input_file': input_file, 'period': period}

print(parse_report_arguments(['report.py', 'sales.csv', '2026-09']))
print(parse_report_arguments(['report.py', 'sales.csv']))

### Running the same idea in a script
Save the following as `report.py`, then execute it from a terminal.

```python
import sys

if len(sys.argv) != 3:
    print('Usage: python report.py <input_file> <period>')
    raise SystemExit(1)

input_file = sys.argv[1]
period = sys.argv[2]
print(f'Preparing report for {input_file} and {period}.')
```

### Instructor note
In a notebook, `sys.argv` contains arguments used to start Jupyter itself. Use simulated lists here so every exercise remains deterministic and does not depend on the environment.

## 3. Packages and reproducible environments

Third-party packages are installed using `pip`. A virtual environment keeps the package versions for one project separate from the versions required by other projects.

### Typical terminal workflow

```bash
python -m venv .venv
# macOS/Linux
source .venv/bin/activate
# Windows PowerShell
.venv\Scripts\Activate.ps1

python -m pip install requests pytest
python -m pip freeze > requirements.txt
python -m pip install -r requirements.txt
```

### Good habits
- Use `python -m pip` rather than relying on a standalone `pip` command, so package installation uses the intended Python interpreter.
- Keep `.venv/` out of Git with `.gitignore`.
- Commit a dependency list such as `requirements.txt`.
- Restart a notebook kernel after installing a package into its environment.
- Never paste secrets, API keys, tokens, or passwords into a notebook committed to Git.

## 4. Web APIs and JSON

An **API** (Application Programming Interface) lets software request data or actions from another service through defined rules. A common web API pattern is:

1. Your program sends an HTTP request to an endpoint.
2. The server sends an HTTP response.
3. The response includes a status code, headers, and often data.
4. Many data APIs return JSON, which maps naturally to Python dictionaries and lists.

| Status code | Meaning | Typical response |
|---|---|---|
| 200 | Success | Read and validate the returned data |
| 400 | Bad request | Check parameters and request format |
| 401 / 403 | Authentication or permission issue | Check credentials and access rights |
| 404 | Resource not found | Check endpoint or identifier |
| 429 | Rate limit exceeded | Slow down and follow API guidance |
| 500+ | Server issue | Retry cautiously or contact the provider |

Always read an API’s documentation, usage limits, terms, authentication rules, and data-license requirements before using it in a project.

### JSON encoding and decoding
`json.loads()` converts a JSON text string into Python objects. `json.dumps()` converts Python objects into JSON text.

JSON supports objects, arrays, strings, numbers, booleans, and null. These correspond approximately to Python dictionaries, lists, strings, numbers, booleans, and `None`.

In [ ]:
import json

json_text = '''
{
  "dataset": "weekly_sales",
  "records": [
    {"week": "2026-W01", "revenue": 1250.50},
    {"week": "2026-W02", "revenue": 1425.75}
  ]
}
'''

payload = json.loads(json_text)
print(type(payload))
print(payload['dataset'])
print(payload['records'][0]['revenue'])

formatted_json = json.dumps(payload, indent=2)
print(formatted_json)

## Code-Along 1.3.1 — Retrieve and inspect API-style data

### Goal
Learn the safe request pattern with `requests`: set a timeout, inspect the status, raise HTTP errors, parse JSON, and validate expected fields.

### Instructor transcript
Say: *External systems are not under our control. A request can time out, return an error status, change its schema, or provide incomplete data. Therefore, API code must check assumptions at each step.*

The live request cell is optional because network access and endpoint availability vary by environment. The simulation cell directly below it is fully runnable and teaches the same parsing logic.

In [ ]:
# Optional live API example. Uncomment only when requests is installed and network access is permitted.
# import requests
#
# endpoint = 'https://itunes.apple.com/search'
# parameters = {'term': 'data science', 'media': 'podcast', 'limit': 5}
#
# try:
#     response = requests.get(endpoint, params=parameters, timeout=15)
#     response.raise_for_status()
# except requests.RequestException as error:
#     print(f'Request failed: {error}')
# else:
#     payload = response.json()
#     for item in payload.get('results', []):
#         print(item.get('collectionName', 'Untitled'))

In [ ]:
# Notebook-safe simulation of an API JSON response.
api_payload = {
    'resultCount': 3,
    'results': [
        {
            'collectionName': 'Practical Data Science',
            'artistName': 'Data Learning Network',
            'primaryGenreName': 'Education'
        },
        {
            'collectionName': 'Analytics in Practice',
            'artistName': 'Evidence Studio',
            'primaryGenreName': 'Technology'
        },
        {
            'collectionName': 'Model Building Weekly',
            'artistName': 'ML Workshop',
            'primaryGenreName': 'Science'
        }
    ]
}

print('Result count:', api_payload.get('resultCount', 0))

for item in api_payload.get('results', []):
    title = item.get('collectionName', 'Untitled')
    creator = item.get('artistName', 'Unknown creator')
    genre = item.get('primaryGenreName', 'Uncategorised')
    print(f'- {title} | {creator} | {genre}')

### Challenge
Write a function named `summarise_results(payload)` that returns a list of formatted strings from `payload['results']`.

Requirements:
- Use `.get()` with sensible defaults for optional fields.
- Return an empty list if `results` is missing.
- Do not print inside the function; return the list instead.

**Stretch:** Return only results matching a selected genre.

In [ ]:
# Your code here
# def summarise_results(payload):
#     ...

## 5. Testing: checking code automatically

An automated test verifies that a function produces the expected result for a chosen input. Tests reduce the chance that a refactor silently breaks behavior you previously relied on.

Good test cases include:
- A normal expected case.
- A boundary case, such as zero, an empty collection, or an exact threshold.
- An invalid case where your function is designed to return a safe response or raise a specific error.

The built-in `assert` statement raises `AssertionError` if its condition is false. In a notebook, assertions provide a lightweight way to demonstrate tests.

In [ ]:
def calculate_discounted_price(price, discount_rate=0.0):
    """Return a discounted price after validating simple business rules."""
    if price < 0:
        raise ValueError('Price cannot be negative.')
    if not 0 <= discount_rate <= 1:
        raise ValueError('Discount rate must be from 0 through 1.')
    return price * (1 - discount_rate)

print(calculate_discounted_price(100, 0.20))
print(calculate_discounted_price(100))

In [ ]:
# Normal cases
assert calculate_discounted_price(100, 0.20) == 80.0
assert calculate_discounted_price(100) == 100.0

# Boundary cases
assert calculate_discounted_price(0, 0.50) == 0.0
assert calculate_discounted_price(100, 0) == 100
assert calculate_discounted_price(100, 1) == 0

print('All assertion tests passed.')

In [ ]:
# Testing expected errors with try / except in a notebook.
try:
    calculate_discounted_price(-1, 0.10)
except ValueError as error:
    assert str(error) == 'Price cannot be negative.'
else:
    raise AssertionError('Expected ValueError for negative price.')

try:
    calculate_discounted_price(100, 1.5)
except ValueError as error:
    assert str(error) == 'Discount rate must be from 0 through 1.'
else:
    raise AssertionError('Expected ValueError for an invalid discount rate.')

print('Expected-error tests passed.')

## Code-Along 1.3.2 — Write a pytest suite

### Goal
Move from notebook assertions to a reusable test file that `pytest` can discover and run.

### Project structure
```text
project/
├── calculations.py
├── test_calculations.py
├── requirements.txt
└── .gitignore
```

Save this function in `calculations.py`:

```python
def calculate_discounted_price(price, discount_rate=0.0):
    if price < 0:
        raise ValueError('Price cannot be negative.')
    if not 0 <= discount_rate <= 1:
        raise ValueError('Discount rate must be from 0 through 1.')
    return price * (1 - discount_rate)
```

Then save this in `test_calculations.py`:

```python
import pytest
from calculations import calculate_discounted_price

def test_discounted_price_with_discount():
    assert calculate_discounted_price(100, 0.20) == 80.0

def test_discounted_price_with_default_discount():
    assert calculate_discounted_price(100) == 100.0

def test_discounted_price_rejects_negative_price():
    with pytest.raises(ValueError, match='Price cannot be negative'):
        calculate_discounted_price(-1, 0.20)

def test_discounted_price_rejects_invalid_rate():
    with pytest.raises(ValueError, match='Discount rate must be from 0 through 1'):
        calculate_discounted_price(100, 1.20)
```

Run the suite from the terminal:

```bash
python -m pytest -q
```

By convention, pytest discovers files named `test_*.py` and functions named `test_*`.

### Challenge
Write a function called `mean_or_none(values)` that:
- Returns the arithmetic mean for a non-empty list of numbers.
- Returns `None` for an empty list.

Write at least three tests: a normal case, a one-value case, and the empty-list case.

**Stretch:** Decide how your function should respond when a list contains non-numeric values, document the decision, and test it.

In [ ]:
# Your code here
# def mean_or_none(values):
#     ...
#
# assert mean_or_none([2, 4, 6]) == 4
# assert mean_or_none([10]) == 10
# assert mean_or_none([]) is None

## Independent practice — A small data retrieval workflow

Use the simulated payload below to create a reliable mini workflow.

1. Write `extract_revenues(payload)` to return valid numeric revenue values.
2. Ignore records with missing or invalid revenue values, but count them.
3. Write `calculate_mean(values)` that returns `None` for an empty list.
4. Write assertions for your expected output.
5. Print a concise quality report.

This exercise combines nested data, safe parsing, reusable functions, and testing.

In [ ]:
sales_payload = {
    'records': [
        {'month': '2026-01', 'revenue': '1250.50'},
        {'month': '2026-02', 'revenue': 'unknown'},
        {'month': '2026-03', 'revenue': '1425.00'},
        {'month': '2026-04'},
        {'month': '2026-05', 'revenue': '1510.25'}
    ]
}

# Your code here
# def extract_revenues(payload):
#     ...
#
# def calculate_mean(values):
#     ...

## Debugging and professional habits

### API troubleshooting
- Confirm the endpoint, parameters, and authentication requirements in the API documentation.
- Set a timeout so a request does not wait indefinitely.
- Check `response.status_code` or call `response.raise_for_status()`.
- Inspect a small sample of the JSON response before assuming its structure.
- Use `.get()` for fields that may be optional.
- Respect rate limits and never expose credentials in a public repository.

### Testing troubleshooting
- Keep functions small and give each function one clear responsibility.
- Write tests that describe behavior rather than implementation details.
- Include normal, boundary, and invalid cases.
- When a test fails, compare expected and actual values before changing either one.
- Run tests after changes, not only at the end of a project.

## Exit ticket
Answer in your own words:

1. What is the difference between a module and a package?
2. Why is a random seed useful in a data-science workflow?
3. Why should a script validate the length of `sys.argv` before accessing an argument?
4. What should you check before assuming that an API response has the fields you expect?
5. Name one boundary case and one invalid case for a function that calculates an average.
6. Why should a project record its dependencies?

## Preview: Module 1.4
Next, you will work with persistent files, CSV data, regular expressions, and practical object-oriented programming.